# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Method:** this cell makes sure DuckDB is version 1.1 or newer, the first version whose Hugging Face login can read the token from the environment.


In [ ]:
%pip -q install "duckdb>=1.1"

## Setup

**Method, step by step:**

1. **Finds the project folder** and **reads the Hugging Face token**, as in the data contract: environment variable or `.env` file, then a Colab Secret named `HF_TOKEN`, then a typed prompt as the last resort. DuckDB reads the token from the environment, so it never appears in a query or an output.
2. **Loads the tools.** `duckdb` queries the warehouse, `pandas` and `numpy` handle the results, `scipy`'s negative binomial is the luck test, and `matplotlib` draws the one chart.
3. **Defines the months and the cache.** The same three development pairs as the data contract; June 2026 is never read. Heavy queries go through `cached()`, which saves each result once under `work/outputs/w03_cache/` and reuses it on later runs.
4. **Defines `show()` and `show_table()`**, which display every result as a titled table.


In [ ]:
import getpass
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import nbinom

# Locally, walk up to the project's top folder (the one holding data/raw and .env); Colab keeps its own folder.
if "google.colab" not in sys.modules:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")


def load_env_file(path: str = ".env") -> None:
    """Copy KEY=value lines from a local .env file into the environment, keeping any value already set."""
    if not os.path.exists(path):
        return
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            key, sep, value = line.strip().partition("=")
            if sep and key and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip())


def read_colab_secret(name: str) -> str | None:
    """Return the Colab Secret called `name`, or None when it cannot be read.

    Outside Colab the import fails. On Colab the secret can be missing, not shared with this
    notebook, or the request can time out. Each case returns None so the caller can ask for the token.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return None
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return None


# Token order: environment variable or .env file, then Colab Secret, then a typed prompt as the last resort.
load_env_file()
token_source = "environment variable or .env file"
if not os.environ.get("HF_TOKEN"):
    secret = read_colab_secret("HF_TOKEN")
    token_source = "Colab Secret" if secret else "typed prompt"
    os.environ["HF_TOKEN"] = secret or getpass.getpass("Hugging Face READ token (hf_...): ")

# DuckDB reads the token from the environment, so it never appears in a query or an output.
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
# Cap DuckDB's memory so large scans spill to disk, and hide its progress bars, which clutter saved outputs.
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY_90D = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
DAILY_FOOTERS = f"read_parquet('{REL}/fact_content_daily_performance/*/*.parquet')"
PAIRS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05")]
DEV_MONTHS = sorted({month for pair in PAIRS for month in pair})
PANEL_MONTHS = list(pd.period_range("2025-01", "2026-05", freq="M").astype(str))
assert "2026-06" not in DEV_MONTHS + PANEL_MONTHS, "The sealed month must never be read."


def month_files(months: list[str]) -> str:
    """Return a DuckDB source that reads exactly the listed monthly partitions of the daily table."""
    paths = ", ".join(f"'{REL}/fact_content_daily_performance/month={month}/*.parquet'" for month in months)
    return f"read_parquet([{paths}])"


CACHE = "work/outputs/w03_cache"
os.makedirs(CACHE, exist_ok=True)


def cached(name: str, sql: str) -> pd.DataFrame:
    """Run a warehouse query once, save its result as Parquet in the cache folder, and reuse it afterwards.

    Repeated scans of the hosted files are slow and can hit rate limits, so every heavy query goes
    through here. Delete the cache folder to force a fresh scan.
    """
    path = f"{CACHE}/{name}.parquet"
    if not os.path.exists(path):
        # Write under a temporary name first, so an interrupted run never leaves a half-written cache file.
        con.execute(f"COPY ({sql}) TO '{path}.tmp' (FORMAT parquet)")
        os.replace(f"{path}.tmp", path)
    return con.sql(f"SELECT * FROM read_parquet('{path}')").df()


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Connected to the warehouse", {
    "token read from": token_source,
    "token shown anywhere": "no (DuckDB reads it from the environment)",
    "development pairs": ", ".join(f"{a} → {b}" for a, b in PAIRS),
    "sealed pair, never read": "2026-05 → 2026-06",
    "cache folder": CACHE,
})

## Data

The signals are tested on the same three development pairs as the data contract, rebuilt with its rules. June 2026 stays sealed.


**Method:** this cell builds the page-month table for February to May 2026 (one row per page per month, position pieces only from days with a valid position, GA4 numbers only where `ga4_data_available IS TRUE`), now including GA4 engaged sessions for the engagement reason code. Each month is scanned once and cached. It also reads each client's search-history start date and each page's content type, search intent, and latest optimization date.


In [ ]:
# One row per page per month, with GA4 engaged sessions added for the engagement reason code.
PAGE_MONTH_SQL = """
    SELECT '{month}' AS month, client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_impressions ELSE 0 END) AS position_impressions,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN gsc_sum_position * gsc_avg_position ELSE 0 END) AS sum_position_sq,
           SUM(CASE WHEN gsc_avg_position >= 1 THEN 0 ELSE gsc_clicks END) AS clicks_without_position,
           COUNT(*) AS days_with_impressions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS ga4_sessions,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions
    FROM {source}
    WHERE gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id"""
page_months = pd.concat([
    cached(f"page_month_v3_{month}", PAGE_MONTH_SQL.format(month=month, source=month_files([month]))) for month in DEV_MONTHS
], ignore_index=True)
assert not page_months.duplicated(["month", "content_hash_id"]).any(), "A page appears twice in one month."

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM {DIM_CLIENTS}").df()
clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
content = con.sql(f"SELECT content_hash_id, content_type, main_intent, last_optimized_date FROM {DIM_CONTENT}").df()
show_table("Inputs loaded", pd.DataFrame({
    "Rows": [f"{len(page_months):,}", f"{len(clients):,}", f"{len(content):,}"],
    "What it is": ["Page-months, February to May 2026", "Clients with their search-history start date",
                   "Pages with content type, search intent, and latest optimization date"],
}, index=pd.Index(["page_months", "clients", "content"], name="Table")))

**Method:** this cell applies the data contract's rules to each development pair (impressions in both months, client history covering all of month M, a valid position in both months, not optimized during the pair). For each month it computes the impression-weighted position, its daily spread, expected clicks from the fine position bins and from the five tiers (both client-scaled), and whether the page fell short by the locked rule: clicks at most half of expected, with negative binomial p below 0.05. The spread factor α is estimated exactly as the data contract does (one estimate per pair from pages whose position barely moved, the median rounded to three decimals), so every notebook shares one target. The month M+1 version of the rule is the target.


In [ ]:
# Rules and bins exactly as in the data contract; the spread factor is estimated below with its method.
BIN_EDGES = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 20.5, 50.5]
BIN_NAMES = [str(position) for position in range(1, 11)] + ["11-20", "21-50", "50+"]


def position_bin(position: pd.Series) -> pd.Series:
    """Return each page's position bin: exact rank up to 10, then 11-20, 21-50, and 50+."""
    return pd.Series(np.array(BIN_NAMES)[np.searchsorted(BIN_EDGES, position, side="right")], index=position.index)


def position_tier(position: pd.Series) -> pd.Series:
    """Return the five tiers used by the earlier score: below 3, below 10, below 20, below 50, and 50 or beyond."""
    tiers = np.select([position < 3, position < 10, position < 20, position < 50], ["top_3", "page_1", "striking", "page_3_5"], "deep")
    return pd.Series(tiers, index=position.index)


def client_aware_expected(frame: pd.DataFrame, clicks: str, impressions: str, group: pd.Series) -> pd.Series:
    """Return expected clicks from each group's pooled click rate, scaled by the page's client factor (+1 on both sides)."""
    rate = frame[clicks].groupby(group).sum() / frame[impressions].groupby(group).sum()
    expected = frame[impressions] * group.map(rate)
    by_client = frame.assign(_expected=expected).groupby("client_hash_id")
    factor = (by_client[clicks].sum() + 1) / (by_client["_expected"].sum() + 1)
    return expected * frame["client_hash_id"].map(factor)


def covered_clients(month_m: str) -> pd.Series:
    """Return the clients whose search history starts on or before the first day of month M."""
    return clients.loc[clients["gsc_data_start"] <= pd.Timestamp(f"{month_m}-01"), "client_hash_id"]


def add_month_measures(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add position, its spread, click-through, and both expectations for one month."""
    impressions, clicks = rows[f"impressions_{suffix}"], rows[f"clicks_{suffix}"]
    rows[f"position_{suffix}"] = rows[f"sum_position_{suffix}"] / rows[f"position_impressions_{suffix}"]
    spread = rows[f"sum_position_sq_{suffix}"] / rows[f"position_impressions_{suffix}"] - rows[f"position_{suffix}"] ** 2
    rows[f"position_volatility_{suffix}"] = np.sqrt(spread.clip(lower=0))
    rows[f"ctr_{suffix}"] = clicks / impressions
    rows[f"expected_bins_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_bin(rows[f"position_{suffix}"]))
    rows[f"expected_tier_{suffix}"] = client_aware_expected(rows, f"clicks_{suffix}", f"impressions_{suffix}", position_tier(rows[f"position_{suffix}"]))
    return rows


def add_luck_test(rows: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Add the negative binomial luck test and the locked shortfall rule for one month."""
    clicks, expected = rows[f"clicks_{suffix}"], rows[f"expected_bins_{suffix}"]
    rows[f"p_{suffix}"] = nbinom.cdf(clicks, 1 / ALPHA, 1 / (1 + ALPHA * expected))
    rows[f"shortfall_{suffix}"] = (rows[f"p_{suffix}"] < 0.05) & (clicks <= 0.5 * expected)
    return rows


def build_pair(month_m: str, month_next: str) -> pd.DataFrame:
    """Return one month pair's eligible rows with both months' measures and the locked target."""
    keys = ["client_hash_id", "content_hash_id"]
    first = page_months[page_months["month"] == month_m].set_index(keys)
    second = page_months[page_months["month"] == month_next].set_index(keys)
    rows = first.join(second, lsuffix="_m", rsuffix="_next", how="inner").reset_index()
    start, end = pd.Timestamp(f"{month_m}-01"), pd.Timestamp(f"{month_next}-01") + pd.offsets.MonthEnd(0)
    rows = rows[rows["client_hash_id"].isin(covered_clients(month_m))]
    rows = rows[(rows["position_impressions_m"] > 0) & (rows["position_impressions_next"] > 0)]
    rows = rows.merge(content, on="content_hash_id", how="left")
    rows = rows[~pd.to_datetime(rows["last_optimized_date"]).between(start, end)].copy()
    rows = add_month_measures(add_month_measures(rows, "m"), "next")
    rows["pair_end"] = end
    return rows.reset_index(drop=True)


def stable_pages(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Return pages whose position moved less than one place, and each one's expected share of clicks in month M+1.

    The expected share corrects for the overall change in click rate between the two months.
    """
    stable = frame[(frame["position_next"] - frame["position_m"]).abs() < 1]
    shift = (stable["clicks_next"].sum() / stable["impressions_next"].sum()) / (stable["clicks_m"].sum() / stable["impressions_m"].sum())
    share = shift * stable["impressions_next"] / (stable["impressions_m"] + shift * stable["impressions_next"])
    return stable, share



def spread_factor(frame: pd.DataFrame) -> float:
    """Return the method-of-moments estimate of the negative binomial spread factor alpha.

    For pages with a stable position, the gap between month-M+1 clicks and the clicks their month-M rate
    predicts has variance (Poisson part) + alpha x (squared clicks). Pooling the excess over the Poisson
    part across pages gives alpha.
    """
    stable, share = stable_pages(frame)
    scale = share / (1 - share)
    gap = stable["clicks_next"] - scale * stable["clicks_m"]
    excess = (gap ** 2 - stable["clicks_next"] - scale ** 2 * stable["clicks_m"]).sum()
    squared = ((stable["clicks_next"] ** 2 - stable["clicks_next"]) + scale ** 2 * (stable["clicks_m"] ** 2 - stable["clicks_m"])).sum()
    return float(excess / squared)



pairs = {f"{month_m} → {month_next}": build_pair(month_m, month_next) for month_m, month_next in PAIRS}
ALPHA = round(float(np.median([spread_factor(rows) for rows in pairs.values()])), 3)
for rows in pairs.values():
    add_luck_test(add_luck_test(rows, "m"), "next")
    rows["target"] = rows["shortfall_next"]
    rows["missed_next"] = (rows["expected_bins_next"] - rows["clicks_next"]).clip(lower=0).where(rows["target"], 0)
show_table(f"Development pairs with the locked target (α = {ALPHA:.3f})", pd.DataFrame({
    "Rows": [f"{len(rows):,}" for rows in pairs.values()],
    "Positives": [f"{rows['target'].sum():,}" for rows in pairs.values()],
    "Base rate": [f"{rows['target'].mean():.1%}" for rows in pairs.values()],
}, index=pd.Index(list(pairs), name="Pair")))

## 1. Distributions

Web traffic is almost always heavy-tailed, and that decides how every test below has to be built.

**Method:** this cell describes the month-M fields across every development row: the share of zeros, the percentiles, and the ratio of the mean to the median (far above 1 means a heavy tail). It then measures how much of all impressions and clicks the top 1% of pages hold, and how the rows split by search intent.


In [ ]:
# Percentiles and tail weight of the month-M fields, across every row of the three pairs.
stacked = pd.concat(pairs.values(), ignore_index=True)
FIELDS = {"impressions_m": "Impressions", "clicks_m": "Clicks", "ctr_m": "Click-through rate",
          "position_m": "Position", "position_volatility_m": "Daily position spread"}
QUANTILES = [0.25, 0.5, 0.75, 0.9, 0.99]


def describe(values: pd.Series, as_percent: bool) -> dict[str, str]:
    """Return the share of zeros, the main percentiles, the maximum, and the mean-to-median ratio of one field."""
    fmt = "{:.2%}".format if as_percent else "{:,.1f}".format
    q = values.quantile(QUANTILES)
    out = {"Zero": f"{(values == 0).mean():.1%}"}
    out.update({label: fmt(q[level]) for level, label in zip(QUANTILES, ["25th", "Median", "75th", "90th", "99th"], strict=True)})
    out["Max"] = fmt(values.max())
    out["Mean ÷ median"] = f"{values.mean() / q[0.5]:.1f}x" if q[0.5] > 0 else "median is 0"
    return out


show_table(f"Month-M fields across all {len(stacked):,} development rows",
           pd.DataFrame({label: describe(stacked[field], field == "ctr_m") for field, label in FIELDS.items()}).T.rename_axis("Field"))


def top_share(values: pd.Series, top: float = 0.01) -> float:
    """Return the share of the total held by the largest `top` fraction of rows."""
    ordered = values.sort_values(ascending=False)
    return float(ordered.iloc[: max(1, int(len(ordered) * top))].sum() / ordered.sum())


intents = stacked["main_intent"].fillna("unknown").value_counts()
show("How concentrated the traffic is", {
    "impressions held by the top 1% of rows": f"{top_share(stacked['impressions_m']):.1%}",
    "clicks held by the top 1% of rows": f"{top_share(stacked['clicks_m']):.1%}",
    **{f"rows with {intent} intent": f"{count:,} ({count / len(stacked):.1%})" for intent, count in intents.items()},
})

## 2. Signal test #1 / #2 / #3 (verdict each)

Every test follows the same shape: the claim in one sentence, a table on a defined slice with the number of pages beside every rate, and a verdict.

**The verdict rule, fixed before any result:** each comparison cell gives a ratio (the group of interest against the comparison group). Only cells with at least 50 pages on both sides count. A ratio of 1.1 or more counts as "higher", 0.9 or less as "lower", anything in between as "no difference". **CONFIRMED** means every counted cell moves the way the claim says; **OPPOSITE** means every counted cell moves the other way; **FALSE** means no counted cell shows a difference; **MIXED** is everything else. Pages under 100 impressions are left out of the size-banded tests: at that size real shortfalls are almost impossible to detect (the leakage check measured 0.1% to 0.3%), so their rates say nothing.

**Method:** this cell defines the size bands, the floor, and the verdict rule as code, so every test applies them identically.


In [ ]:
# The size bands, the sample-size floor, and the verdict rule shared by every test.
SIZE_EDGES = [100, 1_000, 10_000, np.inf]
SIZE_NAMES = ["100 to 999", "1,000 to 9,999", "10,000 or more"]
FLOOR = 50


def size_band(impressions: pd.Series) -> pd.Series:
    """Return each page's size band from its impressions in month M (pages under 100 get no band)."""
    return pd.cut(impressions, SIZE_EDGES, right=False, labels=SIZE_NAMES)


def verdict(effects: list[float], claim: str) -> str:
    """Return the verdict from effect ratios of cells that passed the floor; `claim` is 'higher' or 'lower'."""
    if not effects:
        return "INSUFFICIENT DATA"
    higher, lower = [e >= 1.1 for e in effects], [e <= 0.9 for e in effects]
    agree, against = (higher, lower) if claim == "higher" else (lower, higher)
    if all(agree):
        return "CONFIRMED"
    if all(against):
        return "OPPOSITE"
    if not any(agree) and not any(against):
        return "FALSE"
    return "MIXED"


def rate_cells(name: str, cell: pd.DataFrame, group_of_interest: pd.Series, labels: tuple[str, str]) -> tuple[dict, float | None]:
    """Compare next-month shortfall rates of two groups in one cell; return a table row and the ratio if the floor is met."""
    yes, no = cell[group_of_interest], cell[~group_of_interest]
    rate_yes, rate_no = yes["target"].mean(), no["target"].mean()
    enough = len(yes) >= FLOOR and len(no) >= FLOOR and rate_no > 0
    row = {f"Pages, {labels[0]}": f"{len(yes):,}", f"Next-month rate, {labels[0]}": f"{rate_yes:.1%}",
           f"Pages, {labels[1]}": f"{len(no):,}", f"Next-month rate, {labels[1]}": f"{rate_no:.1%}",
           "Lift": f"{rate_yes / rate_no:.1f}x" if enough else "below floor"}
    return row, (rate_yes / rate_no if enough else None)

### Signal #1: shortfalls persist

**Claim:** a page that falls clearly short of its expected clicks this month is far more likely to fall short again next month, at the same size. The whole project rests on this: if shortfalls did not persist, ranking by this month's shortfall would be pointless.

**Method:** this cell splits each size band of each pair into pages that fell short in month M (by the same rule as the target) and pages that did not, and compares how often each group falls short in month M+1.


In [ ]:
# Next-month shortfall rate for pages short in month M against pages that were not, within each size band.
records, effects = [], []
for name, rows in pairs.items():
    band = size_band(rows["impressions_m"])
    for size in SIZE_NAMES:
        cell = rows[band == size]
        row, effect = rate_cells(name, cell, cell["shortfall_m"], ("short in M", "not short in M"))
        records.append({"Pair": name, "Impressions in month M": size, **row})
        effects += [effect] if effect is not None else []
persist_verdict = verdict(effects, "higher")
show_table("Signal #1: does a shortfall this month predict one next month?",
           pd.DataFrame(records).set_index(["Pair", "Impressions in month M"]))
show("Signal #1 verdict", {"claim": "shortfalls persist into the next month", "cells passing the floor": f"{len(effects)} of {len(records)}",
                           "verdict": persist_verdict})

### Signal #2: search intent changes click-through at the same position

**Claim:** at the same position, keywords with buying intent (commercial or transactional) earn fewer clicks than informational ones, because ads and shopping results compete for the click. If true, a single click-rate curve per position mislabels whole groups of pages, and a learned expectation has a clear job.

**Method:** this cell takes pages with known intent at positions 1 to 20, groups them by position (1 to 3, 4 to 10, 11 to 20) and intent, and reports the pooled click-through rate and clicks ÷ expected. The second measure already accounts for exact position and client, so a gap there is intent's own effect. The verdict compares buying intent with informational intent on clicks ÷ expected.


In [ ]:
# Pooled click-through and clicks ÷ expected by intent, inside three position groups.
INTENT_GROUP = {"commercial": "Buying", "transactional": "Buying", "informational": "Informational", "navigational": "Navigational"}
POSITION_EDGES, POSITION_NAMES = [0, 3.5, 10.5, 20.5], ["1 to 3", "4 to 10", "11 to 20"]
records, effects = [], []
for name, rows in pairs.items():
    known = rows[rows["main_intent"].notna() & (rows["position_m"] < 20.5)]
    summary = known.groupby([pd.cut(known["position_m"], POSITION_EDGES, labels=POSITION_NAMES).rename("position"),
                             known["main_intent"].map(INTENT_GROUP).rename("intent")], observed=True).agg(
        pages=("content_hash_id", "size"), clicks=("clicks_m", "sum"), impressions=("impressions_m", "sum"),
        expected=("expected_bins_m", "sum"))
    summary["ratio"] = summary["clicks"] / summary["expected"]
    for (position, intent), cell in summary.iterrows():
        records.append({"Pair": name, "Position": position, "Intent": intent, "Pages": f"{cell['pages']:,.0f}",
                        "Pooled click-through": f"{cell['clicks'] / cell['impressions']:.2%}", "Clicks ÷ expected": f"{cell['ratio']:.2f}"})
    for position in POSITION_NAMES:
        if (position, "Buying") in summary.index and (position, "Informational") in summary.index:
            buying, informational = summary.loc[(position, "Buying")], summary.loc[(position, "Informational")]
            if min(buying["pages"], informational["pages"]) >= FLOOR:
                effects.append(buying["ratio"] / informational["ratio"])
intent_verdict = verdict(effects, "lower")
show_table("Signal #2: click-through by search intent at the same position",
           pd.DataFrame(records).set_index(["Pair", "Position", "Intent"]))
show("Signal #2 verdict", {"claim": "buying intent clicks less than informational at the same position",
                           "cells passing the floor": f"{len(effects)} of {len(POSITION_NAMES) * len(pairs)}", "verdict": intent_verdict})

### Signal #3: an unstable rank leads to a shortfall

**Claim:** pages whose daily position moves a lot within the month fall short next month more often than stable pages of the same size. This tests the one feature that only daily data makes possible.

**Method:** this cell groups pages by their daily position spread in month M (stable: under 2 places; moderate: 2 to 5; unstable: 5 or more) and compares the next-month shortfall rate of unstable and stable pages within each size band.


In [ ]:
# Next-month shortfall rate by how much the daily position moved in month M, within each size band.
records, effects = [], []
for name, rows in pairs.items():
    band = size_band(rows["impressions_m"])
    spread = rows["position_volatility_m"]
    for size in SIZE_NAMES:
        in_band = band == size
        cell = rows[in_band & ((spread < 2) | (spread >= 5))]
        row, effect = rate_cells(name, cell, cell["position_volatility_m"] >= 5, ("unstable", "stable"))
        moderate = rows[in_band & (spread >= 2) & (spread < 5)]
        records.append({"Pair": name, "Impressions in month M": size, **row,
                        "Pages, moderate": f"{len(moderate):,}", "Next-month rate, moderate": f"{moderate['target'].mean():.1%}"})
        effects += [effect] if effect is not None else []
volatility_verdict = verdict(effects, "higher")
show_table("Signal #3: does an unstable rank predict a shortfall next month?",
           pd.DataFrame(records).set_index(["Pair", "Impressions in month M"]))
show("Signal #3 verdict", {"claim": "unstable pages fall short more often", "cells passing the floor": f"{len(effects)} of {len(records)}",
                           "verdict": volatility_verdict})

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.